# Image Noise: Additive Gaussian & Impulsive Salt-and-Pepper (TP 2 — Exercise 1)

## Assignment Objectives
1. Load `tp1_tp2_cameraman_gray.bmp` as `im1`.
2. Add **zero-mean white Gaussian noise** with variance $\sigma^2 = 100$ ($\sigma = 10$) to obtain `im2`, then study the effect of varying $\sigma^2$.
3. Implement `sp_noise(image, prob)` and apply **salt & pepper** noise with $p = 0.10$ to obtain `im3`, then study the effect of varying $p$.
4. Compare both noise models visually and quantitatively (PSNR).

## Mathematical Principles & Intuition
### Gaussian (additive) noise
Each pixel is perturbed independently:
$$I_{noisy}(y,x) = I(y,x) + n(y,x), \qquad n \sim \mathcal{N}(0, \sigma^2)$$
- The **whole image** is affected — every pixel drifts slightly around its true value.
- The expected MSE is exactly $\sigma^2$, so the expected PSNR is
  $\mathrm{PSNR} = 10\log_{10}(255^2/\sigma^2)$ dB (≈ 28.1 dB for $\sigma^2 = 100$).

### Impulsive (salt & pepper) noise
A random subset of pixels (fraction $p$ overall, $p/2$ each extreme) is **replaced** — not perturbed — by pure black or pure white:
$$I_{noisy}(y,x) = \begin{cases} 0 & \text{with prob. } p/2 \\ 255 & \text{with prob. } p/2 \\ I(y,x) & \text{with prob. } 1-p \end{cases}$$
- $(1-p)$ of the pixels keep their **exact** original value.
- The affected pixels carry the maximal quadratic error $(255)^2$, so PSNR collapses much faster than with Gaussian noise of comparable strength.

> Both generators live in `TAI/utils/noise.py` with **fixed seeds** so that this
> lab, the filtering lab and the Nagao lab all share bit-identical noisy inputs.


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

### What This Cell Does
- Detects Google Colab (`google.colab` in `sys.modules` or `COLAB_RELEASE_TAG`); on Colab it clones `APSD_TAI_LAB` into `/content` (unless already mounted from Drive), so the notebook works even when opened directly from GitHub.
- Locally, it walks upward from the current working directory to the repository root (marker: `README.md` + `APDS/` + `TAI/`) and registers it on `sys.path` — the notebook therefore runs identically from the repo root, from `TAI/…/notebooks/`, or from any sub-folder.
- Verifies `opencv-contrib-python` (checking `cv2.ximgproc`) and installs it only when missing.
- Calls `bootstrap()` from `TAI.utils.env_io`, which resolves `TAI/images/` (the single consolidated, SHA-256 deduplicated dataset) and `TAI/outputs/` (runtime artifacts, never committed).

### Why It Is Needed
- `cv2.imshow()` opens blocking GUI windows and freezes Jupyter kernels — everything here renders through inline Matplotlib instead.
- No hardcoded absolute paths (like `/home/...`): every path is derived from the discovered repository root, so the same cell works on any machine, in any IDE, and on Colab.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "TAI" / "utils").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
else:
    # Local IDE / Jupyter: walk upward to the repository root
    # (identified by README.md + APDS/ + TAI/), whatever the launch directory.
    _cwd = Path.cwd().resolve()
    _repo = next(
        (
            c
            for c in (_cwd, *_cwd.parents)
            if (c / "README.md").is_file() and (c / "APDS").is_dir() and (c / "TAI").is_dir()
        ),
        None,
    )
    if _repo is None:
        raise FileNotFoundError(
            f"APSD_TAI_LAB repository root not found above {_cwd}. "
            "Launch Jupyter from inside the cloned repository."
        )
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))

# Make sure opencv-contrib-python is used (cv2.ximgproc) -- install only if missing
try:
    import cv2

    _has_ximgproc = hasattr(cv2, "ximgproc")
except ImportError:
    _has_ximgproc = False
if not _has_ximgproc:
    %pip install -q opencv-contrib-python
    import cv2

from TAI.utils.env_io import bootstrap, load_image, show_images, plot_image_with_hist

ROOT_DIR, IMG_DIR, OUT_DIR = bootstrap()

import math
import time

import matplotlib.pyplot as plt
import numpy as np

print(f"Repository root : {ROOT_DIR}")
print(f"Images folder   : {IMG_DIR} ({len(list(IMG_DIR.iterdir()))} images)")
print(f"Outputs folder  : {OUT_DIR}")
print(f"OpenCV          : {cv2.__version__} | cv2.ximgproc available: {hasattr(cv2, 'ximgproc')}")


---
## Step 1: `im1`, `im2` (Gaussian $\sigma^2=100$) and `im3` (S&P $p=10\%$)


In [ ]:
from TAI.utils.metrics import calculate_psnr
from TAI.utils.noise import add_gaussian_noise, sp_noise

# 1) Original grayscale image
im1 = load_image("tp1_tp2_cameraman_gray.bmp", cv2.IMREAD_GRAYSCALE)
print(f"Dimensions: {im1.shape}, dtype: {im1.dtype}, range: [{im1.min()}, {im1.max()}]")

# 2) Gaussian noise, variance 100 (sigma = 10)
im2 = add_gaussian_noise(im1, variance=100.0, seed=42)
out_gauss = OUT_DIR / "tp2_exo1_cameraman_gauss_var100.bmp"
cv2.imwrite(str(out_gauss), im2)

# 3) Salt & pepper noise, p = 0.10
im3 = sp_noise(im1, prob=0.10, seed=42)
out_sp = OUT_DIR / "tp2_exo1_cameraman_salt_pepper_p10.bmp"
cv2.imwrite(str(out_sp), im3)

print(f"Saved: {out_gauss}")
print(f"Saved: {out_sp}")
print(f"PSNR(im1, im2 Gaussian var=100) : {calculate_psnr(im1, im2):.2f} dB  (theory ≈ 28.1 dB)")
print(f"PSNR(im1, im3 S&P p=10%)        : {calculate_psnr(im1, im3):.2f} dB")

show_images([
    ("im1 - Original", im1),
    ("im2 - Gaussian var=100", im2),
    ("im3 - Salt & pepper p=10%", im3),
], cols=3)


---
## Step 2: Varying $\sigma^2$ and $p$

| Gaussian variance | Expected PSNR | | S&P probability | Expected PSNR |
|---|---|---|---|---|
| $\sigma^2 = 25$   | 34.1 dB | | $p = 2\%$  | ≈ 21.9 dB |
| $\sigma^2 = 100$  | 28.1 dB | | $p = 5\%$  | ≈ 18.3 dB |
| $\sigma^2 = 400$  | 22.1 dB | | $p = 10\%$ | ≈ 15.1 dB |
| $\sigma^2 = 900$  | 18.59 dB | | $p = 25\%$ | ≈ 10.8 dB |

(Gaussian PSNR from $\mathrm{MSE} = \sigma^2$, i.e. $10\log_{10}(255^2/\sigma^2)$;
impulsive PSNR from $\mathrm{MSE} \approx p \cdot 255^2 / 2$.)


In [ ]:
variances = [25, 100, 400, 900]
probs = [0.02, 0.05, 0.10, 0.25]

gauss_items = [(f"Gaussian var={v} (PSNR={calculate_psnr(im1, add_gaussian_noise(im1, v)):.1f} dB)",
                add_gaussian_noise(im1, v)) for v in variances]
sp_items = [(f"S&P p={p*100:.0f}% (PSNR={calculate_psnr(im1, sp_noise(im1, p)):.1f} dB)",
             sp_noise(im1, p)) for p in probs]

print(f"{'Gaussian var':>12s} | {'measured PSNR (dB)':>18s} | {'theoretical':>12s}")
print("-" * 48)
for v in variances:
    meas = calculate_psnr(im1, add_gaussian_noise(im1, v))
    theo = 10.0 * math.log10(255.0**2 / v)
    print(f"{v:>12d} | {meas:>18.2f} | {theo:>12.2f}")

print(f"\n{'S&P prob':>12s} | {'measured PSNR (dB)':>18s} | {'p*255^2/2 approx':>16s}")
print("-" * 52)
for p in probs:
    meas = calculate_psnr(im1, sp_noise(im1, p))
    theo = 10.0 * math.log10(255.0**2 / (p * 255.0**2 / 2.0))
    print(f"{p:>12.2f} | {meas:>18.2f} | {theo:>16.2f}")

show_images(gauss_items, cols=4)
show_images(sp_items, cols=4)


## Analysis & Key Takeaways
- **Gaussian noise (`im2`)** degrades *every* pixel slightly: fine textures dissolve into uniform grain as $\sigma^2$ grows, while large structures stay readable. Measured PSNR tracks the $10\log_{10}(255^2/\sigma^2)$ law within a fraction of a dB.
- **Impulsive S&P noise (`im3`)** destroys only a fraction $p$ of pixels but with maximal amplitude: at $p = 10\%$ the PSNR (~15 dB) is already worse than Gaussian $\sigma^2 = 900$ (~16 dB), because each impulse contributes the maximal quadratic error $(255)^2$.
- The two models call for **different remedies** (next exercise): the mean/Gaussian family of filters handles Gaussian noise, while the *median* filter is specifically tailored to impulses.
- Deterministic seeds (`seed=42`) make every figure reproducible across this lab, the filtering lab and the Nagao lab — and across pytest runs.
